# Node communities: modularity on any node dendrogram

For a tree whose leaves are the nodes of a network, `ac.modularity_scores(Z, A)` scores every
node of the tree by its contribution to the modularity. The adaptive cut then searches for the
partition of the tree with the highest modularity. The tree can come from scikit-network
(Paris, Louvain hierarchy) or from scipy (here, Ward linkage of a spectral embedding of the
nodes).

This notebook needs `pip install adaptive_cut scikit-network networkx scikit-learn`.

In [1]:
import networkx as nx
import numpy as np
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage
from sklearn.metrics import adjusted_rand_score as ari
from sknetwork.clustering import Louvain, get_modularity
from sknetwork.embedding import Spectral
from sknetwork.hierarchy import LouvainHierarchy, Paris

import adaptive_cut as ac

## Benchmark graphs with planted communities

LFR graphs have communities of different sizes and degrees, with a fraction `mu` of each
node's edges leaving its community. `lfr` returns the adjacency matrix in the format
scikit-network expects (`int32` indices) and the planted community of every node.

In [2]:
def lfr(mu, seed):
    for s in range(seed * 100, seed * 100 + 50):  # the generator sometimes fails; try the next seed
        try:
            G = nx.LFR_benchmark_graph(
                1000,
                2.5,
                1.5,
                mu,
                average_degree=12,
                max_degree=60,
                min_community=15,
                max_community=300,
                seed=s,
            )
            break
        except nx.ExceededMaxIterations:
            continue
    G.remove_edges_from(nx.selfloop_edges(G))
    y = np.empty(len(G), int)
    for c, nodes in enumerate({frozenset(G.nodes[v]["community"]) for v in G}):
        y[list(nodes)] = c
    A = sp.csr_matrix(nx.to_scipy_sparse_array(G, nodelist=range(len(G)), weight=None))
    A.indices, A.indptr = A.indices.astype(np.int32), A.indptr.astype(np.int32)
    return A, y

## Three node trees

scikit-network's `LouvainHierarchy` writes wrong cluster sizes in column 3 of its tree (version
0.33); the tree itself is fine, so `fix_sizes` recomputes that column.

In [3]:
def fix_sizes(Z):
    Z = Z.copy()
    n = len(Z) + 1
    size = np.ones(2 * n - 1)
    for i, (a, b) in enumerate(Z[:, :2].astype(int)):
        size[n + i] = size[a] + size[b]
    Z[:, 3] = size[n:]
    return Z


def spectral_ward(A, k=16):  # Ward linkage of scikit-network's spectral embedding
    return linkage(Spectral(n_components=k).fit_transform(A), "ward")


A, y = lfr(0.1, seed=1)
trees = {
    "Paris": Paris().fit_predict(A),
    "Louvain hierarchy": fix_sizes(LouvainHierarchy(random_state=0).fit_predict(A)),
    "spectral + Ward": spectral_ward(A),
}
louvain = Louvain(random_state=0).fit_predict(A)
print(f"{'Louvain (flat)':>18}: Q={get_modularity(A, louvain):.3f}  ARI {ari(y, louvain):.2f}")
for name, Z in trees.items():
    scores = ac.modularity_scores(Z, A)
    level = ac.best_level_cut(Z, scores)
    res = ac.adaptive_cut(Z, scores, T="auto", rng=0)
    print(
        f"{name:>18}: single-level Q={level.value:.3f} ARI {ari(y, level.labels):.2f} | "
        f"adaptive Q={res.value:.3f} ARI {ari(y, res.labels):.2f}"
    )

    Louvain (flat): Q=0.698  ARI 1.00


             Paris: single-level Q=0.600 ARI 0.71 | adaptive Q=0.600 ARI 0.71
 Louvain hierarchy: single-level Q=0.698 ARI 1.00 | adaptive Q=0.698 ARI 1.00
   spectral + Ward: single-level Q=0.657 ARI 0.77 | adaptive Q=0.698 ARI 0.99


## Several graphs

The spectral tree on ten graphs, against Louvain.

In [4]:
for mu in [0.1, 0.3]:
    for seed in range(5):
        A, y = lfr(mu, seed)
        Z = spectral_ward(A)
        scores = ac.modularity_scores(Z, A)
        level = ac.best_level_cut(Z, scores)
        res = ac.adaptive_cut(Z, scores, T="auto", rng=0)
        louvain = Louvain(random_state=0).fit_predict(A)
        print(
            f"mu={mu} graph {seed}: spectral + Ward, single-level ARI {ari(y, level.labels):.2f} | "
            f"adaptive ARI {ari(y, res.labels):.2f} | Louvain ARI {ari(y, louvain):.2f}"
        )

mu=0.1 graph 0: spectral + Ward, single-level ARI 1.00 | adaptive ARI 1.00 | Louvain ARI 1.00


mu=0.1 graph 1: spectral + Ward, single-level ARI 0.77 | adaptive ARI 0.99 | Louvain ARI 1.00


mu=0.1 graph 2: spectral + Ward, single-level ARI 0.91 | adaptive ARI 0.98 | Louvain ARI 1.00
mu=0.1 graph 3: spectral + Ward, single-level ARI 1.00 | adaptive ARI 1.00 | Louvain ARI 1.00


mu=0.1 graph 4: spectral + Ward, single-level ARI 1.00 | adaptive ARI 1.00 | Louvain ARI 1.00


mu=0.3 graph 0: spectral + Ward, single-level ARI 0.80 | adaptive ARI 0.81 | Louvain ARI 0.95
mu=0.3 graph 1: spectral + Ward, single-level ARI 0.77 | adaptive ARI 0.82 | Louvain ARI 0.88


mu=0.3 graph 2: spectral + Ward, single-level ARI 0.64 | adaptive ARI 0.76 | Louvain ARI 0.84
mu=0.3 graph 3: spectral + Ward, single-level ARI 0.86 | adaptive ARI 0.86 | Louvain ARI 0.93


mu=0.3 graph 4: spectral + Ward, single-level ARI 0.68 | adaptive ARI 0.68 | Louvain ARI 0.75


On trees built by modularity (Paris, Louvain hierarchy) the adaptive cut keeps the best single
level. On the spectral tree it raises the agreement with the planted communities on some
graphs and leaves others unchanged. Louvain optimises modularity directly and is the better
tool when modularity is all you need; the adaptive cut helps when the tree comes from another
method.